# Molecule Explorer: Comparing Chemical Families

In this notebook, we'll compare **families of organic compounds** — alcohols, acids, amines, aromatics, and sugars — building visual intuition for how structure relates to properties.

Just as climate students learn to identify tropical vs. continental climates from heatmaps, you'll learn to recognize chemical families from their descriptor profiles and fingerprints.

## Step 1: Import Libraries

In [ ]:
!pip install rdkit-pypi -q

from rdkit import Chem
from rdkit.Chem import Draw, Descriptors, AllChem, DataStructs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

## Step 2: Build a Compound Library

30 molecules across 5 chemical families.

In [ ]:
compound_library = [
    # Alcohols
    {"name": "Methanol",         "smiles": "CO",                    "family": "Alcohol"},
    {"name": "Ethanol",          "smiles": "CCO",                   "family": "Alcohol"},
    {"name": "1-Propanol",       "smiles": "CCCO",                  "family": "Alcohol"},
    {"name": "1-Butanol",        "smiles": "CCCCO",                 "family": "Alcohol"},
    {"name": "1-Pentanol",       "smiles": "CCCCCO",                "family": "Alcohol"},
    {"name": "Isopropanol",      "smiles": "CC(C)O",                "family": "Alcohol"},
    # Carboxylic Acids
    {"name": "Formic Acid",      "smiles": "O=CO",                  "family": "Acid"},
    {"name": "Acetic Acid",      "smiles": "CC(=O)O",               "family": "Acid"},
    {"name": "Propionic Acid",   "smiles": "CCC(=O)O",              "family": "Acid"},
    {"name": "Butyric Acid",     "smiles": "CCCC(=O)O",             "family": "Acid"},
    {"name": "Pentanoic Acid",   "smiles": "CCCCC(=O)O",            "family": "Acid"},
    {"name": "Benzoic Acid",     "smiles": "OC(=O)c1ccccc1",        "family": "Acid"},
    # Aromatics
    {"name": "Benzene",          "smiles": "c1ccccc1",              "family": "Aromatic"},
    {"name": "Toluene",          "smiles": "Cc1ccccc1",             "family": "Aromatic"},
    {"name": "Naphthalene",      "smiles": "c1ccc2ccccc2c1",        "family": "Aromatic"},
    {"name": "Aniline",          "smiles": "Nc1ccccc1",             "family": "Aromatic"},
    {"name": "Phenol",           "smiles": "Oc1ccccc1",             "family": "Aromatic"},
    {"name": "Styrene",          "smiles": "C=Cc1ccccc1",           "family": "Aromatic"},
    # Amines
    {"name": "Methylamine",      "smiles": "CN",                    "family": "Amine"},
    {"name": "Ethylamine",       "smiles": "CCN",                   "family": "Amine"},
    {"name": "Dimethylamine",    "smiles": "CNC",                   "family": "Amine"},
    {"name": "Trimethylamine",   "smiles": "CN(C)C",                "family": "Amine"},
    {"name": "Propylamine",      "smiles": "CCCN",                  "family": "Amine"},
    {"name": "Diethylamine",     "smiles": "CCNCC",                 "family": "Amine"},
    # Drug-like
    {"name": "Aspirin",          "smiles": "CC(=O)Oc1ccccc1C(=O)O", "family": "Drug"},
    {"name": "Ibuprofen",        "smiles": "CC(C)Cc1ccc(cc1)C(C)C(=O)O", "family": "Drug"},
    {"name": "Acetaminophen",    "smiles": "CC(=O)Nc1ccc(O)cc1",    "family": "Drug"},
    {"name": "Caffeine",         "smiles": "Cn1c(=O)c2c(ncn2C)n(C)c1=O", "family": "Drug"},
    {"name": "Naproxen",         "smiles": "COc1ccc2cc(ccc2c1)C(C)C(=O)O", "family": "Drug"},
    {"name": "Lidocaine",        "smiles": "CCN(CC)CC(=O)Nc1c(C)cccc1C", "family": "Drug"},
]

print(f"Library: {len(compound_library)} compounds in {len(set(c['family'] for c in compound_library))} families")
for fam in ["Alcohol", "Acid", "Aromatic", "Amine", "Drug"]:
    n = sum(1 for c in compound_library if c["family"] == fam)
    print(f"  {fam}: {n} compounds")

## Step 3: Compute Descriptors for All Compounds

In [ ]:
def compute_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return {
        "MW": Descriptors.MolWt(mol),
        "LogP": Descriptors.MolLogP(mol),
        "TPSA": Descriptors.TPSA(mol),
        "HBD": Descriptors.NumHDonors(mol),
        "HBA": Descriptors.NumHAcceptors(mol),
        "RotBonds": Descriptors.NumRotatableBonds(mol),
        "AromaticRings": Descriptors.NumAromaticRings(mol),
        "FractionCSP3": Descriptors.FractionCSP3(mol),
        "HeavyAtoms": mol.GetNumHeavyAtoms(),
    }

rows = []
for c in compound_library:
    desc = compute_descriptors(c["smiles"])
    if desc:
        desc["name"] = c["name"]
        desc["family"] = c["family"]
        desc["smiles"] = c["smiles"]
        rows.append(desc)

df = pd.DataFrame(rows)
df = df.set_index("name")
df.head(10)

## Step 4: Family Portraits — Structure Grids

Draw each chemical family side by side.

In [ ]:
for family in ["Alcohol", "Acid", "Aromatic", "Amine", "Drug"]:
    subset = [c for c in compound_library if c["family"] == family]
    mol_objs = [Chem.MolFromSmiles(c["smiles"]) for c in subset]
    names = [c["name"] for c in subset]
    print(f"\n=== {family}s ===")
    img = Draw.MolsToGridImage(mol_objs, molsPerRow=3, subImgSize=(280, 220), legends=names)
    display(img)

## Step 5: Descriptor Profiles by Family

Compare how molecular properties differ across chemical families.

In [ ]:
family_colors = {
    "Alcohol": "#3498db", "Acid": "#e74c3c", "Aromatic": "#9b59b6",
    "Amine": "#2ecc71", "Drug": "#f39c12"
}

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for ax, prop in zip(axes.flat, ["MW", "LogP", "TPSA", "HBD", "HBA", "RotBonds"]):
    data = [df[df["family"] == fam][prop].values for fam in family_colors]
    bp = ax.boxplot(data, labels=list(family_colors.keys()), patch_artist=True)
    for patch, color in zip(bp["boxes"], family_colors.values()):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    ax.set_title(prop)
    ax.grid(axis="y", alpha=0.3)

plt.suptitle("Molecular Descriptor Distributions by Chemical Family", fontsize=14)
plt.tight_layout()
plt.show()

## Step 6: Fingerprint Heatmap

Visualize the Morgan fingerprints of all 30 compounds — each row is a molecule, each column is a structural feature.

In [ ]:
# Compute fingerprints
fp_data = []
labels = []
colors_list = []

for c in compound_library:
    mol = Chem.MolFromSmiles(c["smiles"])
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=512)
    arr = np.zeros(512)
    DataStructs.ConvertToNumpyArray(fp, arr)
    fp_data.append(arr)
    labels.append(c["name"])
    colors_list.append(family_colors[c["family"]])

fp_matrix = np.array(fp_data)

plt.figure(figsize=(16, 8))
plt.imshow(fp_matrix, aspect="auto", cmap="Blues", interpolation="nearest")
plt.yticks(range(len(labels)), labels, fontsize=7)
plt.xlabel("Fingerprint Bit Index")
plt.title("Morgan Fingerprint Heatmap (30 Compounds)")
plt.colorbar(label="Feature Present")
plt.tight_layout()
plt.show()

print("Notice how compounds in the same family share similar fingerprint patterns.")
print("Aromatics light up different bits than alcohols or amines.")

## Step 7: Chemical Space — MW vs. LogP

A scatter plot of molecular weight vs. LogP is one of the most common views of 'chemical space.'

In [ ]:
plt.figure(figsize=(10, 7))

for family, color in family_colors.items():
    subset = df[df["family"] == family]
    plt.scatter(subset["MW"], subset["LogP"], c=color, s=80,
                label=family, edgecolors="black", linewidth=0.5, zorder=3)
    for name, row in subset.iterrows():
        plt.annotate(name, (row["MW"], row["LogP"]),
                     textcoords="offset points", xytext=(5, 5), fontsize=7, alpha=0.8)

# Lipinski boundaries
plt.axhline(5, color="red", linestyle="--", alpha=0.4, label="Lipinski LogP limit")
plt.axvline(500, color="red", linestyle="--", alpha=0.4, label="Lipinski MW limit")

plt.xlabel("Molecular Weight (g/mol)")
plt.ylabel("LogP")
plt.title("Chemical Space: Molecular Weight vs. LogP")
plt.legend(loc="upper left")
plt.grid(True, alpha=0.3)
plt.show()

print("The red dashed lines show Lipinski's Rule of Five boundaries.")
print("Drug candidates should generally be inside both lines.")

## Step 8: Mystery Molecule Challenge

Below are the descriptor profiles of two **unlabeled** compounds. Can you guess which chemical family they belong to?

In [ ]:
# Mystery molecules
mystery_a = Chem.MolFromSmiles("CCCCCCO")       # 1-Hexanol (Alcohol)
mystery_b = Chem.MolFromSmiles("c1ccc2c(c1)ccc1ccccc12")  # Anthracene (Aromatic)

for label, mol in [("Mystery A", mystery_a), ("Mystery B", mystery_b)]:
    desc = compute_descriptors(Chem.MolToSmiles(mol))
    print(f"\n{label}:")
    print(f"  MW = {desc['MW']:.1f}, LogP = {desc['LogP']:.2f}, TPSA = {desc['TPSA']:.1f}")
    print(f"  Aromatic Rings = {desc['AromaticRings']}, HBD = {desc['HBD']}, FractionCSP3 = {desc['FractionCSP3']:.2f}")
    img = Draw.MolToImage(mol, size=(300, 200))
    display(img)

print("\nBased on the descriptors and structures, which family does each belong to?")
print("Scroll down for the answer...")

In [ ]:
# === ANSWERS ===
print("Mystery A = 1-Hexanol (Alcohol)")
print("  Clue: No aromatic rings, high FractionCSP3, one HBD, moderate LogP")
print()
print("Mystery B = Anthracene (Aromatic)")
print("  Clue: Three aromatic rings, FractionCSP3 = 0, zero HBD, high LogP")

## What's Next

In **Notebook 3**, we'll teach a **neural network** to cluster molecules by chemical family — using the same autoencoder technique from the Birdsong and Climate projects.